# Capstone: Building a Model Comparison Pipeline with diverge

This notebook implements `diverge` — a tool that runs two models on a large prompt set, identifies cases of high disagreement, and clusters them by failure type. This is more actionable than aggregate scores: it tells you exactly where and how Model A beats Model B (and vice versa).

## Why Aggregate Scores Hide the Interesting Signal

A model comparison that shows Model A at 73% and Model B at 71% tells you almost nothing useful. The interesting questions are:
- On which types of tasks does Model B consistently outperform?
- Are the disagreements concentrated in a few failure categories or spread evenly?
- Are Model B's wins on easy tasks that A is underweighted on, or on genuinely hard cases?

`diverge` answers these questions by finding the high-disagreement subset and analyzing it structurally.

## Architecture

The pipeline has four stages:
1. **Parallel evaluation**: run both models on the same prompt set, collect (prompt, response_a, response_b) triples
2. **Judge scoring**: LLM judge scores each response independently on a 1-10 scale
3. **Disagreement filtering**: select cases where score difference exceeds a threshold
4. **Clustering**: group high-disagreement cases by prompt features (length, domain, task type)

In [ ]:
from dataclasses import dataclass, field
from typing import Callable
import random

@dataclass
class DivergeCase:
    prompt: str
    response_a: str
    response_b: str
    score_a: float
    score_b: float
    winner: str
    score_delta: float
    cluster: str = ''

@dataclass
class EvalReport:
    model_a: str
    model_b: str
    n_prompts: int
    n_diverge: int
    a_wins: int
    b_wins: int
    ties: int
    mean_score_a: float
    mean_score_b: float
    diverge_cases: list = field(default_factory=list)

class EvalPipeline:
    def __init__(self, model_a_fn: Callable, model_b_fn: Callable,
                 judge_fn: Callable, model_a_name: str = 'ModelA',
                 model_b_name: str = 'ModelB'):
        self.model_a = model_a_fn
        self.model_b = model_b_fn
        self.judge = judge_fn
        self.a_name = model_a_name
        self.b_name = model_b_name

    def run(self, prompts: list, diverge_threshold: float = 1.5) -> EvalReport:
        all_scores_a, all_scores_b = [], []
        diverge_cases = []
        a_wins = b_wins = ties = 0
        for prompt in prompts:
            resp_a = self.model_a(prompt)
            resp_b = self.model_b(prompt)
            score_a = self.judge(prompt, resp_a)
            score_b = self.judge(prompt, resp_b)
            all_scores_a.append(score_a)
            all_scores_b.append(score_b)
            delta = abs(score_a - score_b)
            winner = 'a' if score_a > score_b else ('b' if score_b > score_a else 'tie')
            if winner == 'a': a_wins += 1
            elif winner == 'b': b_wins += 1
            else: ties += 1
            if delta >= diverge_threshold:
                diverge_cases.append(DivergeCase(
                    prompt=prompt, response_a=resp_a, response_b=resp_b,
                    score_a=score_a, score_b=score_b, winner=winner, score_delta=delta
                ))
        return EvalReport(
            model_a=self.a_name, model_b=self.b_name,
            n_prompts=len(prompts), n_diverge=len(diverge_cases),
            a_wins=a_wins, b_wins=b_wins, ties=ties,
            mean_score_a=sum(all_scores_a)/len(all_scores_a),
            mean_score_b=sum(all_scores_b)/len(all_scores_b),
            diverge_cases=diverge_cases
        )

print('EvalPipeline class defined.')

In [ ]:
def diverge(prompts: list, model_a_fn: Callable, model_b_fn: Callable,
            judge_fn: Callable, threshold: float = 1.5,
            model_a_name: str = 'ModelA', model_b_name: str = 'ModelB') -> EvalReport:
    pipeline = EvalPipeline(model_a_fn, model_b_fn, judge_fn, model_a_name, model_b_name)
    report = pipeline.run(prompts, diverge_threshold=threshold)
    # Simple clustering by prompt length
    for case in report.diverge_cases:
        n = len(case.prompt.split())
        if n < 10:
            case.cluster = 'short_prompt'
        elif n < 30:
            case.cluster = 'medium_prompt'
        else:
            case.cluster = 'long_prompt'
    return report

# Demo: two mock models with different strengths
rng = random.Random(42)

def model_a(prompt):
    # Strong on short prompts
    if len(prompt.split()) < 10:
        return 'Concise and accurate response: ' + prompt[:50]
    return 'A reasonable but not detailed response to: ' + prompt[:50]

def model_b(prompt):
    # Strong on long prompts
    if len(prompt.split()) > 20:
        return 'Detailed response with full context: ' + prompt[:80]
    return 'Short prompt handled adequately by: ' + prompt[:50]

def mock_judge(prompt, response):
    base = 5.0 + len(response)/100
    return min(10, max(1, base + rng.gauss(0, 0.5)))

test_prompts = [
    'What is 2+2?',
    'Name the capital of France.',
    'Explain in detail how neural networks learn to classify images using gradient descent and backpropagation through convolutional layers.',
    'Write a haiku.',
    'What are the key differences between supervised and unsupervised learning methods in machine learning?',
    'Define AI.',
    'Describe the complete process of how a transformer model processes a sequence of tokens through its attention and feed-forward layers to produce output logits.',
    'Is water wet?',
]

report = diverge(test_prompts, model_a, model_b, mock_judge,
                 threshold=1.0, model_a_name='ModelA', model_b_name='ModelB')

print(f'Evaluation Report: {report.model_a} vs {report.model_b}')
print(f'Total prompts:  {report.n_prompts}')
print(f'Diverge cases:  {report.n_diverge} ({report.n_diverge/report.n_prompts:.0%})')
print(f'A wins: {report.a_wins}  B wins: {report.b_wins}  Ties: {report.ties}')
print(f'Mean score A: {report.mean_score_a:.2f}')
print(f'Mean score B: {report.mean_score_b:.2f}')
print()
print('High-divergence cases by cluster:')
clusters = {}
for case in report.diverge_cases:
    clusters.setdefault(case.cluster, []).append(case)
for cluster, cases in sorted(clusters.items()):
    a_w = sum(1 for c in cases if c.winner == 'a')
    b_w = sum(1 for c in cases if c.winner == 'b')
    print(f'  {cluster}: {len(cases)} cases, A wins {a_w}, B wins {b_w}')

## Interpreting diverge Output

The diverge report gives you actionable intelligence:

- If Model B consistently wins on long-prompt cases, it likely has better context utilization or was trained on longer documents
- If Model A wins on short prompts, it may have better instruction-following on simple tasks
- Clusters with one model winning 80%+ indicate a systematic strength/weakness, not random noise

This is the difference between 'Model A scores 2 points higher on our benchmark' (not actionable) and 'Model A is 3x better on complex multi-step reasoning but Model B dominates on concise factual retrieval' (actionable for routing or ensemble decisions).

## Extending to Production Scale

At production scale, diverge runs on:
- 1,000-10,000 prompts sampled from recent production traffic
- Automated clustering using embeddings + k-means rather than simple heuristics
- Longitudinal tracking: run diverge weekly, alert when cluster-level win rates shift
- A/B test infrastructure integration: diverge cases become the evidence base for rollout decisions

This notebook implements the foundation. The `diverge` function and `EvalReport` dataclass are designed to be extended — add your own clustering logic, judge models, and domain-specific metrics on top of this scaffold.

## Series 11 Recap: The LLM Evaluation Stack

Across these 12 notebooks, we built from the ground up:
1. Surface-form metrics (BLEU, ROUGE) and their failure modes
2. LLM-as-judge with bias detection and mitigation
3. Reference benchmarks and contamination detection
4. Human preference evaluation (MT-Bench, Arena ELO)
5. Multi-metric holistic evaluation (HELM, ECE)
6. Custom task evaluation with few-shot sensitivity
7. Factuality evaluation with atomic claim verification
8. Reasoning evaluation with step-level verification
9. Safety evaluation balancing over- and under-refusal
10. Evaluation robustness and Goodhart's Law
11. Agentic trajectory evaluation
12. `diverge`: actionable model comparison infrastructure

The through-line: every metric is a proxy. The job of evaluation engineering is to choose proxies that are hard to game, report uncertainty honestly, and continuously refresh as models improve.